# SPFX-EXTRACTOR V3.1.1
---
**INSTRUCTIONS :**
1. Il faut prévoir un court délai entre l'upload sur le drive et la sync des fichiers sur l'environnement colab.
2. Activez le GPU : Exécution > Modifier le type d'exécution > **GPU T4**.
3. Ajustez vos paramètres dans la Cellule 1.
4. Exécutez toutes les cellules **dans l'ordre**.


**⚠️ NOTE IMPORTANTE SUR L'AUTO-ALIGNEMENT INITIAL :**
En TOUT PREMIER LIEU, avant d'appliquer quelque processing que ce soit, le script regroupe les clips par numéro de prise. Il va ensuite s'assurer d'auto-aligner les clips ensemble (ex: `Lav-01.wav` va s'autoaligner sur `Boom-01.wav`).
Si les clips d'un même groupe ne contiennent pas spécifiquement les mots "Lav" et "Boom" dans leur nom, la piste de référence (la source de cet auto-alignement) sera choisie de façon aléatoire parmi le groupe.

**NOUVEAUTÉS V3.1.1 :**
- **🛡️ Prévol des modèles :** MDX23C, BS-RoFormer et leurs configurations sont validés avant toute purge ou analyse audio. Une ressource indisponible échoue donc immédiatement, jamais après plusieurs heures de calcul.
- **💾 Cache Drive :** les modèles et YAML validés restent dans `PFX_Extractor/0_Model_Cache` entre les sessions Colab.
- **🎚️ Clips ultra-courts :** les rendus de moins de 1024 échantillons sont complétés de silence (21,3 ms maximum) avant l'ancrage BWF, afin d'éviter un WAV vide que Pro Tools ne peut pas importer.

**NOUVEAUTÉS V3.0 :**
- **🎯 Politique Production FX complète :** les 521 classes YAMNet sont réparties par indices exacts entre conservation PFX, retrait humain, retrait d'ambiance, retrait hors-PFX, traitement contextuel et exclusion des masques directs. Les recherches fragiles par sous-chaînes sont supprimées.
- **🎚️ Architecture multi-masques :** humains, ambiances et contenu hors-PFX utilisent des seuils et enveloppes distincts; les pas, vêtements, props, impacts et outils bénéficient d'une protection PFX prioritaire et plafonnée.
- **🧠 Contexte temporel :** animaux et tonalités électroniques sont conservés lorsqu'ils sont ponctuels/foreground et retirés lorsqu'ils deviennent diffus ou persistants.
- **🧱 Robustesse structurelle :** paramètres `audio-separator` validés, conventions de lag corrigées, fréquences d'échantillonnage harmonisées et fusions Snowball redécoupées vers chaque fichier source avec son timecode.
- **💾 Mémoire maîtrisée :** les masques YAMNet restent au pas du modèle et sont interpolés par blocs de 30 secondes, avec diagnostics des classes dominantes et du gain appliqué.

**HISTORIQUE V2.1 :**
- **🛡️ Garde-fou de confiance sur l'auto-alignement :** une corrélation insuffisante empêche désormais l'alignement forcé de prises sans rapport.
- **🗑️ Version de comparaison retirée :** chaque clip ne produit qu'un seul fichier `_PFX_Ready.wav`.

**NOUVEAUTÉS V2.0 (rappel) :**
- **🧠 Détecteur YAMNet indépendant (Cellule 5) :** remplace le sidechain V1.22 basé sur le stem "Vocals" (inefficace — confirmé sur du matériel réel que les modèles de séparation ne détectent quasiment aucune trace vocale sur un soupir). YAMNet est un classificateur d'événements sonores (521 classes AudioSet) totalement indépendant du split vocal/instrumental.
- **👟 Protection Bodytalk :** identifie les classes de mouvement physique (ex: *Walk, footsteps*) pour protéger ces zones du denoise/duck.
- **🔊 Anti-pumping scènes bruyantes :** `DENOISE_ADAPTATIF` (pré-denoise non-stationnaire) et `AI_OVERLAP` relevé (masque IA plus lissé dans le temps).
- **🧹 `dsp_local.py` n'est plus dans la boucle :** tout le traitement est centralisé dans ce notebook.


In [ ]:
# 1. 🎛️ PANNEAU DE CONTRÔLE UTILISATEUR
#@title Ajustement de la Réduction de Bruit (Pre-Denoise)
#@markdown Glissez le curseur pour définir la force du nettoyage avant le passage de l'IA.
#@markdown * 45% = Standard (Préserve la dynamique et la "guenille")
#@markdown * 80%+ = Agressif (Pour les scènes d'autoroute ou industrielles)

NIVEAU_DENOISE_POURCENTAGE = 58 #@param {type:"slider", min:0, max:100, step:1}

#@markdown ---
#@markdown **Pré-denoise adaptatif (anti-pumping)**
#@markdown Un profil de bruit stationnaire matche mal un bruit extérieur qui varie (vent, trafic) —
#@markdown ce mismatch peut faire "pomper" le fond sonore aux endroits où il y avait du dialogue.
#@markdown * Activé (recommandé) = profil de bruit adaptatif (`stationary=False`), plus robuste sur l'extérieur
#@markdown * Désactivé = comportement V1.x (`stationary=True`), peut mieux convenir à un bruit de fond très stable
DENOISE_ADAPTATIF = True #@param {type:"boolean"}

#@markdown ---
#@markdown **Ratio de Fusion IA (RoFormer vs MDX23C)**
#@markdown * 0.50 = 50/50 (Neutre)
#@markdown * 0.60 = Favorise RoFormer (Défaut recommandé pour PFX — splits plus propres, moins d'artefacts)
RATIO_ROFORMER = 0.60 #@param {type:"slider", min:0.1, max:0.9, step:0.05}

#@markdown ---
#@markdown **Chevauchement IA (anti-pumping)**
#@markdown Plus de chevauchement entre segments = masque de séparation plus lissé dans le temps = moins de
#@markdown pumping, au prix d'un peu plus de temps de calcul GPU.
AI_OVERLAP = 12 #@param {type:"slider", min:4, max:16, step:2}

#@markdown ---
#@markdown **Retrait des sons humains (YAMNet)**
#@markdown Atténue voix, vocalisations, respirations, sons physiologiques et applaudissements.
#@markdown Le masque utilise des indices YAMNet exacts, un seuil calibré et une enveloppe temporelle.
#@markdown * 75% = défaut V3.0 équilibré (plage conseillée : 70-80%)
DUCK_DEPTH_HUMAIN = 75 #@param {type:"slider", min:0, max:100, step:5}

#@markdown ---
#@markdown **Retrait des ambiances**
#@markdown Réduit nature/météo, trafic, roomtone, bruit continu et bourdonnements.
#@markdown * 70% = défaut V3.0; le pré-denoise agit déjà en amont (plage conseillée : 60-75%)
DUCK_DEPTH_AMBIANCE = 70 #@param {type:"slider", min:0, max:100, step:5}

#@markdown ---
#@markdown **Retrait du contenu hors PFX**
#@markdown Réduit musique, TV/radio, véhicules, sirènes et graves/vibrations.
#@markdown * 85% = retrait fort sans hard gate (plage conseillée : 80-90%)
DUCK_DEPTH_HORS_PFX = 85 #@param {type:"slider", min:0, max:100, step:5}

#@markdown ---
#@markdown **Protection Production FX**
#@markdown Protège pas, tissu, manipulations, props, impacts et outils contre le pré-denoise et les
#@markdown faux positifs YAMNet. La restauration brute est plafonnée pour ne jamais annuler le nettoyage.
#@markdown * 0% = Aucune protection additionnelle
#@markdown * 70% = défaut V3.0 équilibré (plage conseillée : 60-75%)
PROTECTION_PFX = 70 #@param {type:"slider", min:0, max:90, step:5}

print(f"✅ Niveau de réduction de bruit réglé à : {NIVEAU_DENOISE_POURCENTAGE}% ({'adaptatif' if DENOISE_ADAPTATIF else 'stationnaire'})")
print(f"✅ Ratio IA configuré à : {RATIO_ROFORMER*100:.0f}% RoFormer / {(1-RATIO_ROFORMER)*100:.0f}% MDX23C | Overlap = {AI_OVERLAP}")
print(f"✅ Retrait YAMNet : humains {DUCK_DEPTH_HUMAIN}% | ambiance {DUCK_DEPTH_AMBIANCE}% | hors-PFX {DUCK_DEPTH_HORS_PFX}%")
print(f"✅ Protection PFX : {'Désactivée' if PROTECTION_PFX == 0 else f'{PROTECTION_PFX}%'}")


In [ ]:
# 2. INSTALLATION DES DÉPENDANCES ET VÉRIFICATION DU GPU
print("🚀 Installation de audio-separator et des dépendances...")
!pip install -q "audio-separator[gpu]==0.44.3" soundfile scipy "noisereduce==3.0.3" tensorflow-hub
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install -q onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
from IPython.display import clear_output
from importlib.metadata import version as package_version
clear_output()
print(f"✅ Dépendances installées — audio-separator {package_version('audio-separator')} | noisereduce {package_version('noisereduce')}.")

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Device : {device.upper()}")
if device == 'cpu':
    print("   ⚠️  Aucun GPU — activez le GPU T4 dans Exécution > Modifier le type d'exécution.")
else:
    print(f"   ✅ Carte graphique détectée : {torch.cuda.get_device_name(0)}")


In [ ]:
# 3. MONTAGE DE GOOGLE DRIVE ET DOSSIERS TEMP
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/PFX_Extractor"
FOLDER_IN       = os.path.join(DRIVE_BASE, "1_Bruts_vers_Colab")
FOLDER_OUT      = os.path.join(DRIVE_BASE, "2_Environnements_IA")
FOLDER_MODEL_CACHE = os.path.join(DRIVE_BASE, "0_Model_Cache")

FOLDER_ALIGNED  = "/content/temp_aligned"
FOLDER_MERGED   = "/content/temp_merged"
FOLDER_DENOISED = "/content/temp_denoised"
FOLDER_AI_STEMS = "/content/temp_ai_stems"
FOLDER_TC_READY = "/content/temp_tc_ready"

for folder in [FOLDER_IN, FOLDER_OUT, FOLDER_MODEL_CACHE, FOLDER_ALIGNED, FOLDER_MERGED, FOLDER_DENOISED, FOLDER_AI_STEMS, FOLDER_TC_READY]:
    os.makedirs(folder, exist_ok=True)

print(f"\n📁 Dossiers configurés.")

In [ ]:
# 4. CONFIGURATION DES MODÈLES DE SÉPARATION
import os
import gc
import torch
import inspect
import shutil
import numpy as np
import soundfile as sf
import noisereduce as nr
import subprocess
import re
import time
import requests
import yaml
from scipy import signal
from audio_separator.separator import Separator

MODEL_CACHE_DIR = FOLDER_MODEL_CACHE
os.makedirs(MODEL_CACHE_DIR,      exist_ok=True)
os.makedirs('/content/temp_out', exist_ok=True)

MODEL_MDX23C = 'MDX23C-8KFFT-InstVoc_HQ.ckpt'
MODEL_ROFORMER = 'model_bs_roformer_ep_317_sdr_12.9755.ckpt'

MDXC_PARAMS = {'segment_size': 256, 'batch_size': 1, 'overlap': AI_OVERLAP}
ROFORMER_PARAMS = {'segment_size': 256, 'override_model_segment_size': True, 'batch_size': 1, 'overlap': AI_OVERLAP}

TEMP_IN  = "/content/temp_in.wav"
TEMP_OUT = "/content/temp_out"
MIN_SAFE_DURATION = 5.0
# ffmpeg peut ecrire un chunk data vide pour un rendu BWF de quelques echantillons.
# 1024 echantillons a 48 kHz = 21,3 ms : imperceptible, mais importable et valide.
MIN_EXPORT_SAMPLES = 1024

# Sources de configuration verifiees independamment de l'URL UVR obsolete.
MODEL_YAML_SPECS = {
    MODEL_MDX23C: {
        'filename': 'model_2_stem_full_band_8k.yaml',
        'url': 'https://raw.githubusercontent.com/TRvlvr/application_data/main/mdx_model_data/mdx_c_configs/model_2_stem_full_band_8k.yaml',
        'required': {('audio', 'sample_rate'): 44100, ('audio', 'n_fft'): 8192},
    },
    MODEL_ROFORMER: {
        'filename': 'model_bs_roformer_ep_317_sdr_12.9755.yaml',
        'url': 'https://raw.githubusercontent.com/ZFTurbo/Music-Source-Separation-Training/main/configs/viperx/model_bs_roformer_ep_317_sdr_12.9755.yaml',
        'required': {('audio', 'sample_rate'): 44100, ('model', 'dim'): 512},
    },
}

def _yaml_is_valid(path, required_values):
    try:
        data = yaml.load(path.read_text(encoding='utf-8'), Loader=yaml.FullLoader)
        if not isinstance(data, dict):
            return False
        for (section, key), expected in required_values.items():
            if data.get(section, {}).get(key) != expected:
                return False
        return True
    except Exception:
        return False

def _download_yaml_with_retry(url, destination, required_values, attempts=3):
    last_error = None
    for attempt in range(1, attempts + 1):
        temp_path = destination.with_suffix(destination.suffix + '.part')
        try:
            print(f'   Telechargement configuration ({attempt}/{attempts}) : {destination.name}')
            response = requests.get(url, timeout=(15, 90))
            response.raise_for_status()
            temp_path.write_bytes(response.content)
            if not _yaml_is_valid(temp_path, required_values):
                raise RuntimeError('contenu YAML invalide ou mauvais modele')
            temp_path.replace(destination)
            return
        except Exception as exc:
            last_error = exc
            if temp_path.exists():
                temp_path.unlink()
            if attempt < attempts:
                time.sleep(3 * attempt)
    raise RuntimeError(f'Impossible de preparer {destination.name} apres {attempts} essais : {last_error}')

def preflight_models():
    """Prepare et charge tous les modeles avant le moindre calcul audio."""
    from pathlib import Path
    cache = Path(MODEL_CACHE_DIR)
    print('\n' + '=' * 50)
    print('PREFLIGHT MODELES - aucun audio ne sera traite avant validation')
    print('=' * 50)

    for model_name, spec in MODEL_YAML_SPECS.items():
        yaml_path = cache / spec['filename']
        if _yaml_is_valid(yaml_path, spec['required']):
            print(f'   YAML valide en cache : {yaml_path.name}')
        else:
            _download_yaml_with_retry(spec['url'], yaml_path, spec['required'])
            print(f'   YAML valide et mis en cache : {yaml_path.name}')

    for model_name, params in ((MODEL_MDX23C, MDXC_PARAMS), (MODEL_ROFORMER, ROFORMER_PARAMS)):
        print(f'   Verification du modele : {model_name}')
        separator = Separator(
            log_level=30, model_file_dir=MODEL_CACHE_DIR, output_dir=TEMP_OUT,
            output_format='WAV', normalization_threshold=1.0,
            mdxc_params=dict(params),
        )
        separator.load_model(model_name)
        del separator
        gc.collect()
        torch.cuda.empty_cache()

    print('PREFLIGHT REUSSI : modeles, YAML et GPU valides. Le traitement audio peut commencer.\n')

print("✅ Configuration terminée.")


In [ ]:
# 5. DÉTECTEUR YAMNET — TAXONOMIE PFX EXACTE ET MASQUES SÉPARÉS
"""
Les 521 classes AudioSet sont partitionnées par indices exacts. Aucune recherche par
sous-chaîne n'est autorisée : elle confondait notamment Hum/Humming et Run avec des
classes sans rapport. Les masques humain, ambiance, hors-PFX et protection PFX ont
des seuils et des enveloppes temporelles distincts.
"""
import tensorflow as tf
import tensorflow_hub as hub
import csv

print("📥 Chargement du modèle YAMNet (détecteur d'événements sonores)...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

class_map_path = yamnet_model.class_map_path().numpy().decode('utf-8')
class_names = []
with tf.io.gfile.GFile(class_map_path) as f:
    reader = csv.DictReader(f)
    for row in reader:
        class_names.append(row['display_name'])

if len(class_names) != 521:
    raise RuntimeError(f"Taxonomie YAMNet inattendue : {len(class_names)} classes au lieu de 521.")
YAMNET_SENTINEL_NAMES = {
    0: 'Speech', 58: 'Clapping', 62: 'Applause', 132: 'Music', 277: 'Wind',
    294: 'Vehicle', 321: 'Traffic noise, roadway noise', 390: 'Siren',
    458: 'Chorus effect', 487: 'Rumble', 490: 'Hum', 494: 'Silence',
    495: 'Sine wave', 500: 'Inside, small room', 518: 'Television',
    520: 'Field recording',
}
sentinel_mismatches = {
    index: (expected, class_names[index])
    for index, expected in YAMNET_SENTINEL_NAMES.items()
    if class_names[index] != expected
}
if sentinel_mismatches:
    raise RuntimeError(f"Ordre de taxonomie YAMNet incompatible : {sentinel_mismatches}")

def _inclusive(start, end):
    return set(range(start, end + 1))

HUMAN_CLASS_SET = (
    _inclusive(0, 45)
    | _inclusive(49, 55)
    | {59, 60, 61, 62, 63, 64, 65, 66}
)
PFX_CLASS_SET = (
    {46, 47, 48, 56, 57, 58}
    | _inclusive(348, 389)
    | {392, 393, 394}
    | _inclusive(398, 457)
    | _inclusive(459, 486)
    | {488, 489, 491, 492, 493, 498, 499}
)
AMBIENCE_CLASS_SET = (
    _inclusive(277, 293)
    | {321, 490, 507, 508, 509, 510, 513, 514, 515}
)
OUTSIDE_PFX_CLASS_SET = (
    _inclusive(132, 276)
    | (_inclusive(294, 347) - {321})
    | {390, 391, 395, 396, 397, 458, 487, 511, 512, 516, 517, 518, 519}
)
ANIMAL_CONTEXT_CLASS_SET = _inclusive(67, 131)
TONAL_CONTEXT_CLASS_SET = {495, 496, 497}
CONTEXT_CLASS_SET = ANIMAL_CONTEXT_CLASS_SET | TONAL_CONTEXT_CLASS_SET
EXCLUDED_DIRECT_CLASS_SET = {494} | _inclusive(500, 506) | {520}

ANIMAL_FOREGROUND_CLASS_SET = {
    69, 70, 71, 72, 73, 76, 77, 78, 79, 80, 82, 83, 84, 85, 86, 88,
    89, 90, 91, 92, 94, 95, 96, 97, 98, 99, 100, 101, 102, 104, 105,
    112, 113, 114, 115, 127, 128, 129, 130,
}
ANIMAL_DIFFUSE_CLASS_SET = ANIMAL_CONTEXT_CLASS_SET - ANIMAL_FOREGROUND_CLASS_SET

YAMNET_CLASS_GROUPS = {
    'CONSERVER_PFX': PFX_CLASS_SET,
    'RETIRER_HUMAIN': HUMAN_CLASS_SET,
    'RETIRER_AMBIANCE': AMBIENCE_CLASS_SET,
    'RETIRER_HORS_PFX': OUTSIDE_PFX_CLASS_SET,
    'TRAITER_CONTEXTE': CONTEXT_CLASS_SET,
    'EXCLURE_MASQUE_DIRECT': EXCLUDED_DIRECT_CLASS_SET,
}
EXPECTED_GROUP_COUNTS = {
    'CONSERVER_PFX': 146,
    'RETIRER_HUMAIN': 61,
    'RETIRER_AMBIANCE': 26,
    'RETIRER_HORS_PFX': 211,
    'TRAITER_CONTEXTE': 68,
    'EXCLURE_MASQUE_DIRECT': 9,
}

assigned = set()
for group_name, group_set in YAMNET_CLASS_GROUPS.items():
    overlap = assigned & group_set
    if overlap:
        raise RuntimeError(f"Indices YAMNet présents dans plusieurs groupes : {sorted(overlap)}")
    assigned |= group_set
    if len(group_set) != EXPECTED_GROUP_COUNTS[group_name]:
        raise RuntimeError(
            f"Compte invalide pour {group_name}: {len(group_set)} au lieu de "
            f"{EXPECTED_GROUP_COUNTS[group_name]}."
        )
if assigned != set(range(521)):
    missing = sorted(set(range(521)) - assigned)
    extra = sorted(assigned - set(range(521)))
    raise RuntimeError(f"Partition YAMNet incomplète — absents={missing}, hors plage={extra}")

for group_name, group_set in YAMNET_CLASS_GROUPS.items():
    print(f"   • {group_name}: {len(group_set)} classes")
print("   ✅ Audit taxonomie : 521/521 indices exacts, aucun chevauchement.")

HUMAN_CLASS_IDX = sorted(HUMAN_CLASS_SET)
PFX_CLASS_IDX = sorted(PFX_CLASS_SET)
AMBIENCE_CLASS_IDX = sorted(AMBIENCE_CLASS_SET)
OUTSIDE_PFX_CLASS_IDX = sorted(OUTSIDE_PFX_CLASS_SET)
ANIMAL_CONTEXT_CLASS_IDX = sorted(ANIMAL_CONTEXT_CLASS_SET)
ANIMAL_FOREGROUND_CLASS_IDX = sorted(ANIMAL_FOREGROUND_CLASS_SET)
ANIMAL_DIFFUSE_CLASS_IDX = sorted(ANIMAL_DIFFUSE_CLASS_SET)
TONAL_CONTEXT_CLASS_IDX = sorted(TONAL_CONTEXT_CLASS_SET)
EXCLUDED_DIRECT_CLASS_IDX = sorted(EXCLUDED_DIRECT_CLASS_SET)

YAMNET_FRAME_HOP_S = 0.48
YAMNET_MASK_BLOCK_S = 30.0
YAMNET_THRESHOLDS = {
    'human': (0.18, 0.45),
    'pfx': (0.12, 0.35),
    'ambience': (0.12, 0.38),
    'outside_pfx': (0.15, 0.42),
    'context': (0.14, 0.40),
}

def _max_class_score(scores, indices):
    if scores.shape[0] == 0 or not indices:
        return np.zeros(scores.shape[0], dtype=np.float32)
    return np.max(scores[:, indices], axis=1).astype(np.float32)

def _soft_threshold(values, low, high):
    if not 0.0 <= low < high <= 1.0:
        raise ValueError(f"Seuil YAMNet invalide : {low}, {high}")
    return np.clip((np.asarray(values, dtype=np.float32) - low) / (high - low), 0.0, 1.0)

def _moving_mean_frames(values, window_s):
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return values.copy()
    width = max(1, int(round(window_s / YAMNET_FRAME_HOP_S)))
    if width == 1:
        return values.copy()
    left = width // 2
    right = width - 1 - left
    padded = np.pad(values, (left, right), mode='edge')
    kernel = np.ones(width, dtype=np.float32) / width
    return np.convolve(padded, kernel, mode='valid').astype(np.float32)

def _attack_release_frames(values, attack_s, release_s):
    values = np.clip(np.asarray(values, dtype=np.float32), 0.0, 1.0)
    if values.size == 0:
        return values.copy()
    attack_alpha = np.exp(-YAMNET_FRAME_HOP_S / max(float(attack_s), 1e-3))
    release_alpha = np.exp(-YAMNET_FRAME_HOP_S / max(float(release_s), 1e-3))
    envelope = np.empty_like(values)
    previous = 0.0
    for i, target in enumerate(values):
        alpha = attack_alpha if target > previous else release_alpha
        previous = alpha * previous + (1.0 - alpha) * float(target)
        envelope[i] = previous
    return np.clip(envelope, 0.0, 1.0).astype(np.float32)

def yamnet_mask_block(yamnet_data, mask_name, start, end, target_sr):
    if yamnet_data is None or end <= start:
        return np.zeros(max(0, end - start), dtype=np.float32)
    frame_values = np.asarray(yamnet_data.get('frames', {}).get(mask_name, []), dtype=np.float32)
    if frame_values.size == 0:
        return np.zeros(end - start, dtype=np.float32)
    frame_times = np.arange(frame_values.size, dtype=np.float64) * YAMNET_FRAME_HOP_S + YAMNET_FRAME_HOP_S
    sample_times = np.arange(start, end, dtype=np.float64) / float(target_sr)
    return np.interp(
        sample_times, frame_times, frame_values,
        left=float(frame_values[0]), right=float(frame_values[-1]),
    ).astype(np.float32)

def analyze_yamnet(filepath):
    audio, sr = sf.read(filepath, always_2d=True)
    mono = np.mean(audio, axis=1).astype(np.float32)

    if int(sr) == 48000:
        mono_16k = signal.resample_poly(mono, up=1, down=3).astype(np.float32)
    else:
        divisor = int(np.gcd(int(sr), 16000))
        mono_16k = signal.resample_poly(
            mono, up=16000 // divisor, down=int(sr) // divisor
        ).astype(np.float32)

    peak = float(np.max(np.abs(mono_16k))) if mono_16k.size else 0.0
    if peak > 1.0:
        mono_16k = mono_16k / peak

    if mono_16k.size == 0:
        return {
            'frames': {name: np.zeros(0, dtype=np.float32) for name in ('human', 'ambience', 'outside_pfx', 'pfx')},
            'source_sr': int(sr), 'source_length': 0, 'top_classes': [],
        }

    scores, _embeddings, _spectrogram = yamnet_model(mono_16k)
    scores = scores.numpy().astype(np.float32)
    n_frames = scores.shape[0]
    if n_frames == 0:
        return {
            'frames': {name: np.zeros(0, dtype=np.float32) for name in ('human', 'ambience', 'outside_pfx', 'pfx')},
            'source_sr': int(sr), 'source_length': len(mono), 'top_classes': [],
        }

    human_raw = _max_class_score(scores, HUMAN_CLASS_IDX)
    pfx_raw = _max_class_score(scores, PFX_CLASS_IDX)
    ambience_raw = _max_class_score(scores, AMBIENCE_CLASS_IDX)
    outside_raw = _max_class_score(scores, OUTSIDE_PFX_CLASS_IDX)
    animal_raw = _max_class_score(scores, ANIMAL_CONTEXT_CLASS_IDX)
    animal_foreground_raw = _max_class_score(scores, ANIMAL_FOREGROUND_CLASS_IDX)
    animal_diffuse_raw = _max_class_score(scores, ANIMAL_DIFFUSE_CLASS_IDX)
    tonal_raw = _max_class_score(scores, TONAL_CONTEXT_CLASS_IDX)

    human_base = _soft_threshold(human_raw, *YAMNET_THRESHOLDS['human'])
    pfx_base = _soft_threshold(pfx_raw, *YAMNET_THRESHOLDS['pfx'])
    ambience_base = _soft_threshold(ambience_raw, *YAMNET_THRESHOLDS['ambience'])
    outside_base = _soft_threshold(outside_raw, *YAMNET_THRESHOLDS['outside_pfx'])

    animal_activity = _soft_threshold(animal_raw, *YAMNET_THRESHOLDS['context'])
    animal_foreground = _soft_threshold(animal_foreground_raw, *YAMNET_THRESHOLDS['context'])
    animal_diffuse = _soft_threshold(animal_diffuse_raw, *YAMNET_THRESHOLDS['context'])
    animal_persistence = _soft_threshold(_moving_mean_frames(animal_activity, 4.0), 0.30, 0.70)
    diffuse_persistence = _soft_threshold(_moving_mean_frames(animal_diffuse, 2.5), 0.20, 0.60)
    animal_remove = np.maximum(
        animal_activity * animal_persistence,
        animal_diffuse * diffuse_persistence,
    )
    animal_keep = animal_foreground * (1.0 - animal_persistence)

    tonal_activity = _soft_threshold(tonal_raw, *YAMNET_THRESHOLDS['context'])
    tonal_persistence = _soft_threshold(_moving_mean_frames(tonal_activity, 2.5), 0.35, 0.75)
    tonal_remove = tonal_activity * tonal_persistence
    tonal_keep = tonal_activity * (1.0 - tonal_persistence)

    frame_masks = {
        'human': _attack_release_frames(human_base, attack_s=0.12, release_s=0.80),
        'ambience': _attack_release_frames(
            np.maximum(ambience_base, animal_remove), attack_s=0.45, release_s=3.50
        ),
        'outside_pfx': _attack_release_frames(
            np.maximum(outside_base, tonal_remove), attack_s=0.20, release_s=1.20
        ),
        'pfx': _attack_release_frames(
            np.maximum.reduce([pfx_base, animal_keep, tonal_keep]), attack_s=0.05, release_s=0.60
        ),
    }

    per_class_peak = np.max(scores, axis=0)
    top_indices = np.argsort(per_class_peak)[-8:][::-1]
    top_classes = [(int(i), class_names[i], float(per_class_peak[i])) for i in top_indices]
    active = {name: 100.0 * float(np.mean(mask > 0.10)) for name, mask in frame_masks.items()}
    print(
        f"      Masques actifs — humain {active['human']:.1f}% | ambiance {active['ambience']:.1f}% | "
        f"hors-PFX {active['outside_pfx']:.1f}% | protection PFX {active['pfx']:.1f}%"
    )
    print("      Top YAMNet : " + ", ".join(f"#{index} {name}={score:.2f}" for index, name, score in top_classes[:5]))

    return {
        'frames': frame_masks,
        'source_sr': int(sr),
        'source_length': len(mono),
        'top_classes': top_classes,
    }

print("\n✅ Détecteur YAMNet PFX prêt.")


In [ ]:
# 6. MOTEUR DSP, TIMECODE ET ALGORITHMES STRUCTURELS
ALIGNMENT_CONFIDENCE_MIN = 0.15  # en dessous : corrélation jugée non fiable -> pas d'alignement forcé

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def initial_mic_alignment(files_list):
    print(f"\n{'='*50}")
    print(f"🎯 ÉTAPE 0.1 : AUTO-ALIGNEMENT INITIAL (LAV -> BOOM)")
    print(f"{'='*50}")

    take_groups = {}
    for f in files_list:
        nums = re.findall(r'\d+', f)
        take = nums[-1] if nums else f
        if take not in take_groups: take_groups[take] = []
        take_groups[take].append(f)

    aligned_files = []
    for take, files in take_groups.items():
        if len(files) == 1:
            shutil.copy(os.path.join(FOLDER_IN, files[0]), os.path.join(FOLDER_ALIGNED, files[0]))
            aligned_files.append(files[0])
            continue

        ref_file = next((x for x in files if 'boom' in x.lower()), files[0])
        ref_path = os.path.join(FOLDER_IN, ref_file)
        shutil.copy(ref_path, os.path.join(FOLDER_ALIGNED, ref_file))
        aligned_files.append(ref_file)

        print(f"   🎙️ Groupe [Take {take}] : Référence -> {ref_file}")

        ref_audio, sr = sf.read(ref_path, always_2d=True)
        ref_mono = np.mean(ref_audio, axis=1)
        max_samples = 30 * sr

        for tgt_file in files:
            if tgt_file == ref_file: continue
            print(f"      🧲 Alignement de la cible : {tgt_file}")
            tgt_path = os.path.join(FOLDER_IN, tgt_file)
            tgt_audio, _ = sf.read(tgt_path, always_2d=True)
            tgt_mono = np.mean(tgt_audio, axis=1)

            if np.max(np.abs(ref_mono)) < 1e-6 or np.max(np.abs(tgt_mono)) < 1e-6:
                sf.write(os.path.join(FOLDER_ALIGNED, tgt_file), tgt_audio, sr, subtype='FLOAT')
                aligned_files.append(tgt_file)
                continue

            ref_seg = ref_mono[:max_samples]
            tgt_seg = tgt_mono[:max_samples]
            corr = signal.correlate(ref_seg, tgt_seg, mode='full', method='fft')
            peak_idx = int(np.argmax(np.abs(corr)))
            lag = peak_idx - (len(tgt_seg) - 1)

            norm = np.linalg.norm(ref_seg) * np.linalg.norm(tgt_seg)
            confidence = abs(corr[peak_idx]) / norm if norm > 1e-9 else 0.0

            if confidence < ALIGNMENT_CONFIDENCE_MIN:
                print(f"      ⚠️ Corrélation trop faible ({confidence:.2f} < {ALIGNMENT_CONFIDENCE_MIN}) — probablement pas la même prise, alignement ignoré (copie brute).")
                sf.write(os.path.join(FOLDER_ALIGNED, tgt_file), tgt_audio, sr, subtype='FLOAT')
                aligned_files.append(tgt_file)
                continue

            # scipy.signal.correlate(ref, cible) retourne un lag positif lorsque
            # la cible est en avance : il faut donc la retarder (et inversement).
            tgt_aligned = np.zeros_like(tgt_audio)
            if lag > 0:
                shift = int(lag)
                length = min(len(tgt_audio), len(tgt_aligned) - shift)
                if length > 0:
                    tgt_aligned[shift:shift+length] = tgt_audio[:length]
            elif lag < 0:
                shift = int(-lag)
                length = min(len(tgt_aligned), len(tgt_audio) - shift)
                if length > 0:
                    tgt_aligned[:length] = tgt_audio[shift:shift+length]
            else:
                tgt_aligned = tgt_audio.copy()

            print(f"         ✅ Confiance {confidence:.2f} — lag={lag} samples")
            sf.write(os.path.join(FOLDER_ALIGNED, tgt_file), tgt_aligned, sr, subtype='FLOAT')
            aligned_files.append(tgt_file)

    return aligned_files

def snowball_merge(files_list):
    print(f"\n{'='*50}")
    print(f"☃️  ÉTAPE 0.2 : SNOWBALL MERGE (Séparation par Famille)")
    print(f"{'='*50}")

    families = {}
    for f in files_list:
        match = re.search(r'^(.*)([-_ ]\d+)', f)
        base_name = match.group(1).strip() if match else "Misc"
        if base_name not in families: families[base_name] = []
        families[base_name].append(f)

    merged_groups = {}

    for base_name, fam_files in families.items():
        fam_files = sorted(fam_files, key=natural_sort_key)
        current_group_audio = None
        current_segments = []
        current_duration = 0.0
        family_sr = None
        family_channels = None

        family_anchors = []

        for f in fam_files:
            in_path = os.path.join(FOLDER_ALIGNED, f)
            audio, sr = sf.read(in_path, always_2d=True)
            if family_sr is None:
                family_sr = int(sr)
                family_channels = audio.shape[1]
            elif int(sr) != family_sr:
                print(f"   🔄 Rééchantillonnage Snowball : {f} ({sr} Hz -> {family_sr} Hz)")
                audio = _resample_to_rate(audio, sr, family_sr, axis=0)

            if audio.shape[1] != family_channels:
                raise ValueError(
                    f"Canaux incompatibles dans la famille '{base_name}' : "
                    f"{f} possède {audio.shape[1]} canal(aux), attendu {family_channels}."
                )

            start_sample = 0 if current_group_audio is None else len(current_group_audio)
            segment = {
                'filename': f,
                'start_sample': start_sample,
                'length_samples': len(audio),
            }
            if current_group_audio is None:
                current_group_audio = audio
            else:
                current_group_audio = np.concatenate((current_group_audio, audio), axis=0)
            current_segments.append(segment)
            current_duration = len(current_group_audio) / family_sr

            if current_duration >= MIN_SAFE_DURATION:
                anchor_filename = current_segments[0]['filename']
                out_path = os.path.join(FOLDER_MERGED, anchor_filename)
                sf.write(out_path, current_group_audio, family_sr, subtype='FLOAT')
                merged_groups[anchor_filename] = {
                    'sample_rate': family_sr,
                    'segments': [dict(item) for item in current_segments],
                }
                family_anchors.append(anchor_filename)
                if len(current_segments) > 1:
                    print(f"   🔗 Fusion Famille [{base_name}] : '{anchor_filename}' cuit à {current_duration:.2f}s")
                current_group_audio = None
                current_segments = []
                current_duration = 0.0

        if current_group_audio is not None:
            if not family_anchors:
                 anchor_filename = current_segments[0]['filename']
                 out_path = os.path.join(FOLDER_MERGED, anchor_filename)
                 sf.write(out_path, current_group_audio, family_sr, subtype='FLOAT')
                 merged_groups[anchor_filename] = {
                     'sample_rate': family_sr,
                     'segments': [dict(item) for item in current_segments],
                 }
            else:
                 last_anchor = family_anchors[-1]
                 last_path = os.path.join(FOLDER_MERGED, last_anchor)
                 last_audio, _ = sf.read(last_path, always_2d=True)
                 offset = len(last_audio)
                 combined = np.concatenate((last_audio, current_group_audio), axis=0)
                 sf.write(last_path, combined, family_sr, subtype='FLOAT')
                 trailing_segments = [
                     {**item, 'start_sample': item['start_sample'] + offset}
                     for item in current_segments
                 ]
                 merged_groups[last_anchor]['segments'].extend(trailing_segments)
                 print(f"   🔗 Fusion finale Famille [{base_name}] : Reste greffé à '{last_anchor}'")

    return list(merged_groups.keys()), merged_groups

def get_sample_rate(filepath):
    try:
        _, sr = sf.read(filepath, frames=1)
        return sr
    except: return 48000

def inject_timecode(original_path, processed_path, final_output_path):
    original_sr = get_sample_rate(original_path)
    try:
        probe_cmd = ['ffprobe', '-v', 'error', '-show_entries', 'format_tags=time_reference', '-of', 'default=nw=1:nk=1', original_path]
        time_ref = subprocess.check_output(probe_cmd).decode('utf-8').strip()
    except Exception: time_ref = ""

    cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-i', processed_path, '-i', original_path, '-map', '0:a', '-map_metadata', '1', '-write_bext', '1']
    if time_ref: cmd.extend(['-metadata:g', f'time_reference={time_ref}'])
    cmd.extend(['-ar', str(original_sr), '-c:a', 'pcm_f32le', final_output_path])
    subprocess.run(cmd)

def prep_denoise_batch(files_list, yamnet_cache):
    print(f"\n{'='*50}\n🧹 ÉTAPE 1 : PRÉ-NETTOYAGE OBLIGATOIRE ({NIVEAU_DENOISE_POURCENTAGE}%, {'adaptatif' if DENOISE_ADAPTATIF else 'stationnaire'})\n{'='*50}")

    prop_decrease_val = NIVEAU_DENOISE_POURCENTAGE / 100.0
    stationary_flag = not DENOISE_ADAPTATIF

    for f in files_list:
        in_path = os.path.join(FOLDER_MERGED, f)
        out_path = os.path.join(FOLDER_DENOISED, f)

        print(f"   🚿 Nettoyage chirurgical ({NIVEAU_DENOISE_POURCENTAGE}%) : {f}")
        audio, sr = sf.read(in_path, always_2d=True)
        reduced_audio = np.zeros_like(audio)
        for i in range(audio.shape[1]):
            reduced_channel = nr.reduce_noise(
                y=audio[:, i], sr=sr, stationary=stationary_flag, prop_decrease=prop_decrease_val,
                n_fft=8192, win_length=8192, hop_length=512,
                time_mask_smooth_ms=150, freq_mask_smooth_hz=500
            )
            if np.isnan(reduced_channel).any(): reduced_channel = audio[:, i]
            reduced_audio[:, i] = reduced_channel

        base, _ = os.path.splitext(f)
        yamnet_data = yamnet_cache.get(base)
        if yamnet_data is not None and PROTECTION_PFX > 0:
            block_size = max(1, int(round(sr * YAMNET_MASK_BLOCK_S)))
            active_samples = 0
            protection_depth = min(PROTECTION_PFX / 100.0, 0.90)
            for start in range(0, audio.shape[0], block_size):
                end = min(start + block_size, audio.shape[0])
                pfx_mask = yamnet_mask_block(yamnet_data, 'pfx', start, end, sr)
                restore = np.clip(pfx_mask * protection_depth, 0.0, 0.90)
                active_samples += int(np.count_nonzero(restore > 0.05))
                restore_col = restore.reshape(-1, 1)
                reduced_audio[start:end] = (
                    audio[start:end] * restore_col
                    + reduced_audio[start:end] * (1.0 - restore_col)
                )
            pct = 100.0 * active_samples / max(1, audio.shape[0])
            print(f"      🛡️ Protection PFX : {pct:.1f}% du fichier partiellement restauré")

        sf.write(out_path, reduced_audio, sr, subtype='FLOAT')

def _cleanup_temp():
    for fname in os.listdir(TEMP_OUT):
        fpath = os.path.join(TEMP_OUT, fname)
        if os.path.isfile(fpath):
            try: os.remove(fpath)
            except: pass
    if os.path.exists(TEMP_IN): os.remove(TEMP_IN)

def process_batch(model_name, model_suffix, custom_params, files_list):
    if not files_list: return
    print(f"\n{'='*50}\n⚙️  CHARGEMENT DU MODÈLE : {model_name}\n{'='*50}")
    separator_kwargs = {
        'log_level': 30, 'model_file_dir': MODEL_CACHE_DIR, 'output_dir': '/content/temp_out',
        'output_format': 'WAV', 'normalization_threshold': 1.0,
        # MDX23C et BS-RoFormer utilisent tous deux l'architecture MDXC
        # dans audio-separator 0.44.3.
        'mdxc_params': dict(custom_params),
    }
    sig = inspect.signature(Separator.__init__)
    constructor_params = set(sig.parameters.keys())
    unsupported = sorted(set(separator_kwargs) - constructor_params)
    if unsupported:
        raise RuntimeError(
            "Version audio-separator incompatible : paramètres constructeur absents : "
            + ", ".join(unsupported)
        )
    print(f"   🔧 Paramètres MDXC appliqués : {separator_kwargs['mdxc_params']}")
    separator = Separator(**separator_kwargs)
    separator.load_model(model_name)

    for f in files_list:
        input_path = os.path.join(FOLDER_DENOISED, f)
        base, _ = os.path.splitext(f)
        inst_output_path = os.path.join(FOLDER_AI_STEMS, base + model_suffix)

        print(f"\n   🎵 Traitement : {f}")
        audio, sr = sf.read(input_path, always_2d=True)
        audio_stereo = np.repeat(audio, 2, axis=1) if (audio.shape[1] == 1) else audio
        sf.write(TEMP_IN, audio_stereo, sr, subtype='FLOAT')

        try: res_files = separator.separate(TEMP_IN)
        except Exception as e: res_files = None

        if not res_files:
            print(f"   🔄 FALLBACK : Bypass de l'IA.")
            sf.write(inst_output_path, audio, sr, subtype='FLOAT')
            _cleanup_temp()
            continue

        inst_candidates = [r for r in res_files if 'instrumental' in r.lower()]
        inst_file = inst_candidates[0] if inst_candidates else res_files[0]

        inst_path = os.path.join(TEMP_OUT, inst_file) if not os.path.isabs(inst_file) else inst_file
        if not os.path.exists(inst_path):
            print(f"   🔄 FALLBACK : Fichier silencieux (ignoré par l'IA).")
            sf.write(inst_output_path, audio, sr, subtype='FLOAT')
            _cleanup_temp()
            continue

        inst_audio, inst_sr = sf.read(inst_path, always_2d=True)
        if audio.shape[1] == 1: inst_audio = np.mean(inst_audio, axis=1, keepdims=True)
        sf.write(inst_output_path, inst_audio, inst_sr, subtype='FLOAT')

        _cleanup_temp()

    del separator
    gc.collect()
    torch.cuda.empty_cache()

def _resample_to_rate(data, source_sr, target_sr, axis=0):
    source_sr = int(source_sr)
    target_sr = int(target_sr)
    if source_sr <= 0 or target_sr <= 0:
        raise ValueError(f"Fréquence d'échantillonnage invalide : {source_sr} -> {target_sr}")
    if source_sr == target_sr:
        return np.asarray(data)
    divisor = int(np.gcd(source_sr, target_sr))
    return signal.resample_poly(
        data, up=target_sr // divisor, down=source_sr // divisor, axis=axis
    )

def _apply_yamnet_masks_inplace(mixed, sr, yamnet_data, output_path):
    depths = {
        'human': DUCK_DEPTH_HUMAIN / 100.0,
        'ambience': DUCK_DEPTH_AMBIANCE / 100.0,
        'outside_pfx': DUCK_DEPTH_HORS_PFX / 100.0,
    }
    if yamnet_data is None:
        if any(depth > 0 for depth in depths.values()):
            print(f"      ⚠️ YAMNet [{os.path.basename(output_path)}] : masques absents, gain neutre")
        return mixed
    if not any(depth > 0 for depth in depths.values()):
        return mixed

    block_size = max(1, int(round(sr * YAMNET_MASK_BLOCK_S)))
    protection_depth = PROTECTION_PFX / 100.0
    active_samples = 0
    min_gain = 1.0
    for start in range(0, len(mixed), block_size):
        end = min(start + block_size, len(mixed))
        pfx = yamnet_mask_block(yamnet_data, 'pfx', start, end, sr) * protection_depth
        human = yamnet_mask_block(yamnet_data, 'human', start, end, sr)
        ambience = yamnet_mask_block(yamnet_data, 'ambience', start, end, sr)
        outside = yamnet_mask_block(yamnet_data, 'outside_pfx', start, end, sr)

        # La protection PFX est forte contre l'ambiance, modérée contre les autres retraits.
        human *= 1.0 - 0.35 * pfx
        ambience *= 1.0 - 0.85 * pfx
        outside *= 1.0 - 0.55 * pfx

        reduction = np.maximum.reduce([
            human * depths['human'],
            ambience * depths['ambience'],
            outside * depths['outside_pfx'],
        ])
        gain = np.clip(1.0 - reduction, 0.03, 1.0).astype(np.float32)
        active_samples += int(np.count_nonzero(gain < 0.95))
        min_gain = min(min_gain, float(np.min(gain)))
        mixed[start:end] *= gain.reshape(-1, 1)

    pct_active = 100.0 * active_samples / max(1, len(mixed))
    print(
        f"      🔍 YAMNet DIAG [{os.path.basename(output_path)}] : "
        f"gain min={min_gain:.2f} | {pct_active:.1f}% du fichier atténué"
    )
    return mixed

def auto_align_and_mix(ref_path, target_path, output_path, yamnet_data=None):
    ref_audio, sr = sf.read(ref_path, always_2d=True)
    tgt_audio, tgt_sr = sf.read(target_path, always_2d=True)
    if int(tgt_sr) != int(sr):
        print(f"      🔄 Rééchantillonnage stem cible : {tgt_sr} Hz -> {sr} Hz")
        tgt_audio = _resample_to_rate(tgt_audio, tgt_sr, sr, axis=0)
    ref_mono = np.mean(ref_audio, axis=1)
    tgt_mono = np.mean(tgt_audio, axis=1)
    max_samples = 30 * sr

    if np.max(np.abs(ref_mono)) < 1e-6 or np.max(np.abs(tgt_mono)) < 1e-6:
        tgt_fitted = np.zeros_like(ref_audio)
        length = min(len(ref_audio), len(tgt_audio))
        tgt_fitted[:length] = tgt_audio[:length]
        mixed = RATIO_ROFORMER * ref_audio + (1.0 - RATIO_ROFORMER) * tgt_fitted
    else:
        corr = signal.correlate(ref_mono[:max_samples], tgt_mono[:max_samples], mode='full', method='fft')
        lag = np.argmax(corr) - (len(tgt_mono[:max_samples]) - 1)

        # Même convention de lag que pour l'alignement micro ci-dessus.
        tgt_aligned = np.zeros_like(ref_audio)
        if lag > 0:
            shift = int(lag)
            length = min(len(tgt_audio), len(ref_audio) - shift)
            if length > 0:
                tgt_aligned[shift:shift+length] = tgt_audio[:length]
        elif lag < 0:
            shift = int(-lag)
            length = min(len(ref_audio), len(tgt_audio) - shift)
            if length > 0:
                tgt_aligned[:length] = tgt_audio[shift:shift+length]
        else:
            length = min(len(ref_audio), len(tgt_audio))
            tgt_aligned[:length] = tgt_audio[:length]

        mixed = RATIO_ROFORMER * ref_audio + (1.0 - RATIO_ROFORMER) * tgt_aligned

    mixed = _apply_yamnet_masks_inplace(mixed, sr, yamnet_data, output_path)

    sf.write(output_path, mixed, sr, subtype='FLOAT')

def iter_split_merged_audio(mix_path, group_info):
    mix_audio, mix_sr = sf.read(mix_path, always_2d=True)
    merged_sr = int(group_info['sample_rate'])
    if merged_sr <= 0:
        raise ValueError(f"Fréquence Snowball invalide : {merged_sr}")

    segments = group_info.get('segments', [])
    if not segments:
        raise ValueError(f"Aucun segment Snowball associé à {mix_path}")

    for segment in segments:
        start_sample = int(round(segment['start_sample'] * mix_sr / merged_sr))
        end_source = segment['start_sample'] + segment['length_samples']
        end_sample = int(round(end_source * mix_sr / merged_sr))
        expected_length = end_sample - start_sample
        if expected_length <= 0 or start_sample < 0 or start_sample >= len(mix_audio):
            raise RuntimeError(
                f"Bornes Snowball invalides pour {segment['filename']} : "
                f"{start_sample}:{end_sample} sur {len(mix_audio)} échantillons."
            )

        available_end = min(end_sample, len(mix_audio))
        segment_audio = mix_audio[start_sample:available_end].copy()
        missing = expected_length - len(segment_audio)
        if missing > 0:
            tolerance = max(2, int(round(mix_sr * 0.01)))
            if missing > tolerance:
                raise RuntimeError(
                    f"Sortie IA trop courte pour redécouper {segment['filename']} : "
                    f"{missing} échantillons manquants."
                )
            segment_audio = np.pad(segment_audio, ((0, missing), (0, 0)), mode='constant')

        if len(segment_audio) < MIN_EXPORT_SAMPLES:
            padding = MIN_EXPORT_SAMPLES - len(segment_audio)
            print(
                f"      WARNING: {segment['filename']} ne contient que {len(segment_audio)} echantillon(s); "
                f"ajout de {padding} echantillon(s) de silence pour un BWF importable."
            )
            segment_audio = np.pad(segment_audio, ((0, padding), (0, 0)), mode='constant')

        yield segment['filename'], segment_audio, mix_sr


In [ ]:
# 7. LANCEMENT DU WORKFLOW COMPLET INTELLIGENT
# Le preflight doit reussir AVANT toute purge ou tout calcul couteux.
preflight_models()

print("\n🧹 PURGE DES DOSSIERS TEMPORAIRES (Prévention des fichiers fantômes)...")
for folder in [FOLDER_ALIGNED, FOLDER_MERGED, FOLDER_DENOISED, FOLDER_AI_STEMS, FOLDER_TC_READY]:
    if os.path.exists(folder):
        for tmp_f in os.listdir(folder):
            try: os.remove(os.path.join(folder, tmp_f))
            except: pass

all_files_in = [f for f in os.listdir(FOLDER_IN) if f.lower().endswith('.wav')]

if not all_files_in:
    print("\n📭 Aucun fichier .wav trouvé dans le dossier IN.")
else:
    print(f"\n🔥 {len(all_files_in)} fichier(s) détecté(s). Début du workflow...\n")

    aligned_files_list = initial_mic_alignment(all_files_in)
    merged_files_list, merged_groups = snowball_merge(aligned_files_list)

    print(f"\n{'='*50}\n🧠 ÉTAPE 0.3 : ANALYSE YAMNET MULTI-MASQUES PFX\n{'='*50}")
    yamnet_cache = {}
    for f in merged_files_list:
        base, _ = os.path.splitext(f)
        print(f"   🔎 Analyse : {f}")
        yamnet_cache[base] = analyze_yamnet(os.path.join(FOLDER_MERGED, f))

    prep_denoise_batch(merged_files_list, yamnet_cache)

    process_batch(MODEL_MDX23C, "_MDX23C.wav", MDXC_PARAMS, merged_files_list)
    process_batch(MODEL_ROFORMER, "_RoFormer.wav", ROFORMER_PARAMS, merged_files_list)

    print(f"\n{'='*50}\n🎛️  ÉTAPE 4 : AUTO-ALIGNEMENT IA, MASQUES YAMNET ET ANCRAGE TC BWF\n{'='*50}\n")

    for f in merged_files_list:
        base, _ = os.path.splitext(f)
        mdx_path = os.path.join(FOLDER_AI_STEMS, base + "_MDX23C.wav")
        rof_path = os.path.join(FOLDER_AI_STEMS, base + "_RoFormer.wav")

        mix_path = os.path.join(FOLDER_TC_READY, base + "_Mix_Temp.wav")
        group_info = merged_groups[f]

        if os.path.exists(mdx_path) and os.path.exists(rof_path):
            yamnet_data = yamnet_cache.get(base)
            auto_align_and_mix(
                rof_path, mdx_path, mix_path, yamnet_data=yamnet_data
            )

            if os.path.exists(mix_path):
                print(f"   ✂️ Redécoupage Snowball : {len(group_info['segments'])} clip(s) source")
                for source_filename, segment_audio, segment_sr in iter_split_merged_audio(mix_path, group_info):
                    source_base, _ = os.path.splitext(source_filename)
                    original_in_path = os.path.join(FOLDER_IN, source_filename)
                    split_path = os.path.join(FOLDER_TC_READY, source_base + "_Split_Temp.wav")
                    final_out_path = os.path.join(FOLDER_OUT, source_base + "_PFX_Ready.wav")
                    sf.write(split_path, segment_audio, segment_sr, subtype='FLOAT')
                    print(f"   ⏱️ Ancrage du TC (Source: {source_filename}) pour : {source_base}")
                    inject_timecode(original_in_path, split_path, final_out_path)
                    rendered_info = sf.info(final_out_path)
                    if rendered_info.frames < MIN_EXPORT_SAMPLES:
                        raise RuntimeError(
                            f"Export BWF vide ou trop court pour {source_filename}: "
                            f"{rendered_info.frames} echantillon(s)."
                        )
                    print(f"      ✅ Prêt : {os.path.basename(final_out_path)}")

    print("\n🧹 Nettoyage des dossiers temporaires...")
    for folder in [FOLDER_ALIGNED, FOLDER_MERGED, FOLDER_DENOISED, FOLDER_AI_STEMS, FOLDER_TC_READY]:
        for tmp_f in os.listdir(folder):
            os.remove(os.path.join(folder, tmp_f))

    print("\n🎉 WORKFLOW CONTINU TERMINÉ !")


In [ ]:
# 8. DÉCONNEXION AUTOMATIQUE
import time
import torch
from google.colab import runtime

# --- CALCUL DU COÛT DE LA SESSION ---
try:
    uptime_seconds = float(open('/proc/uptime').read().split()[0])
    uptime_hours = uptime_seconds / 3600.0
    gpu_name = torch.cuda.get_device_name(0)

    # Taux approximatifs de Compute Units par heure sur Colab
    rates = {'T4': 1.96, 'L4': 5.09, 'V100': 5.4, 'A100': 13.08}
    rate = 1.96 # defaut
    for k, v in rates.items():
        if k in gpu_name: rate = v; break

    units_used = uptime_hours * rate
    cost_usd = units_used * (16.0 / 100.0) # 16$ par 100 units

    print("\n" + "="*40)
    print("💰 BILAN FINANCIER DE LA SESSION")
    print("="*40)
    print(f"⏱️ Temps d'exécution total : {uptime_hours:.2f} heures")
    print(f"🖥️ GPU Utilisé : {gpu_name} ({rate} unités/heure)")
    print(f"⚡ Compute Units brûlés : ~{units_used:.2f}")
    print(f"💸 Coût estimé : {cost_usd:.3f} USD")
    print("="*40 + "\n")
except Exception as e:
    print(f"\n⚠️ Impossible de calculer le coût : {e}")

print("⏳ L'instance Colab se déconnectera automatiquement dans 10 secondes...")
time.sleep(10)
print("👋 Déconnexion. Au revoir!")
runtime.unassign()
